# PPI walkthrough

Weights → MinMax → score → ranking → surrogate → uncertainty → Model B.

Run from repo root after `python scripts/train.py`.

In [1]:
import sys
from pathlib import Path
ROOT = Path('/workspace') if Path('/workspace/src/ppi').exists() else Path.cwd().resolve()
if not (ROOT / 'src' / 'ppi').exists():
    ROOT = Path.cwd().resolve().parent if (Path.cwd() / '01_ppi_walkthrough.ipynb').exists() else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

from ppi.io import load_match_stats, DATA_RAW
from ppi.weights import load_weights
from ppi.scoring import compute_ppi, scale_features
from ppi.explain import feature_contributions
from ppi.ranking import rank_players
from ppi.validation import compare_expert_vs_surrogate
from ppi.models import PositionModelBundle
from ppi.uncertainty import bootstrap_ppi_intervals
from ppi.ranking_model import PairwiseRanker, compare_expert_vs_ranker

print('ROOT', ROOT)
MATCH_ID = 'el-clasico-2025-05-11'


ROOT /workspace


## 1. Load match + weights

In [2]:
match = load_match_stats(match_id=MATCH_ID)
weights = load_weights(DATA_RAW / 'importance_weights.csv')
match[['name','profile','goals','assists','playing_time']].head()

,name,profile,goals,assists,playing_time
0,Ferran Torres,Attacker,0,3,88
1,Raphinha,Attacker,2,0,90
2,Lamine Yamal,Attacker,1,0,90
3,Dani Olmo,Midfielder,0,0,76
4,Fermin Lopez,Midfielder,0,0,14


## 2. Expert PPI + ranking

In [3]:
scored, scaler, _ = compute_ppi(match, weights)
ranked = rank_players(scored, min_minutes=30)
ranked

,rank,name,profile,ppi,playing_time
0,1,Raphinha,Attacker,10.000000,90
1,2,Lamine Yamal,Attacker,8.909260,90
2,3,Pedri,Midfielder,8.814809,90
3,4,Wojciech Szczesny,Goalkeeper,7.854922,90
4,5,Frenkie de Jong,Midfielder,6.361388,90
5,6,Ferran Torres,Attacker,5.976543,88
6,7,Inigo Martinez,Defender,4.670659,90
7,8,Dani Olmo,Midfielder,3.874352,76
8,9,Eric Garcia,Defender,3.654057,77
9,10,Alejandro Balde,Defender,2.498473,34


## 3. Feature contributions (Raphinha)

In [4]:
scaled, _ = scale_features(scored, scaler=scaler, fit=False)
row = scored.loc[scored['name'] == 'Raphinha'].iloc[0]
contrib = feature_contributions(scaled.loc[row.name], row['profile'], weights)
contrib.head(8)

,feature,label,weight,scaled_value,contribution,abs_contribution
0,goals,Goals,0.28,1.000000,0.280000,0.280000
1,key_passes,Key passes,0.10,1.000000,0.100000,0.100000
2,shots,Shots,0.08,0.857143,0.068571,0.068571
3,playing_time,Playing time (min),0.06,1.000000,0.060000,0.060000
4,shots_on_goal,Shots on target,0.10,0.400000,0.040000,0.040000
5,crosses,Crosses,0.04,1.000000,0.040000,0.040000
6,possession_participation,Possession share %,0.05,0.629272,0.031464,0.031464
7,offsides,Offsides,-0.03,1.000000,-0.030000,0.030000


## 4. Surrogate vs expert

In [5]:
bundle = PositionModelBundle.load(ROOT / 'models')
compare_expert_vs_surrogate(match, weights, bundle)

,name,profile,expert_ppi,surrogate_ppi,delta
0,Raphinha,Attacker,10.00,9.10,-0.90
1,Lamine Yamal,Attacker,8.91,8.91,0.00
2,Pedri,Midfielder,8.81,8.81,0.00
3,Wojciech Szczesny,Goalkeeper,7.85,6.45,-1.40
4,Frenkie de Jong,Midfielder,6.36,6.45,0.09
5,Ferran Torres,Attacker,5.98,5.44,-0.54
6,Inigo Martinez,Defender,4.67,4.70,0.03
7,Dani Olmo,Midfielder,3.87,4.03,0.16
8,Eric Garcia,Defender,3.65,4.03,0.38
9,Alejandro Balde,Defender,2.50,2.35,-0.15


## 5. Leave-one-match-out metrics

In [6]:
bundle.metrics.get('leave_one_match_out', {}).get('pooled')

{'n': 43.0,
 'mae': 0.8662204136311199,
 'r2': 0.8264635585408762,
 'spearman': 0.9017237814764211}

## 6. Bootstrap uncertainty

In [7]:
intervals = bootstrap_ppi_intervals(match, weights, n_boot=100, min_minutes=30)
intervals.head()

,rank,name,profile,ppi,ppi_lo,ppi_hi,top3_prob,width
0,1,Raphinha,Attacker,10.00,8.48,10.00,1.0,1.52
1,2,Lamine Yamal,Attacker,9.05,7.92,10.00,1.0,2.08
2,3,Pedri,Midfielder,8.89,7.52,10.00,1.0,2.48
3,4,Wojciech Szczesny,Goalkeeper,7.66,6.31,7.81,0.0,1.50
4,5,Frenkie de Jong,Midfielder,6.19,5.08,6.67,0.0,1.59


## 7. Model B pairwise ranks

In [8]:
ranker = PairwiseRanker.load(ROOT / 'models')
compare_expert_vs_ranker(match, weights, ranker) if ranker else 'train first'

,name,profile,ppi,rank_expert,rank_b,rank_score,rank_delta
0,Raphinha,Attacker,10.000000,1,3,0.866688,2
1,Lamine Yamal,Attacker,8.909260,2,6,0.624274,4
2,Pedri,Midfielder,8.814809,3,1,0.995474,-2
3,Wojciech Szczesny,Goalkeeper,7.854922,4,10,0.500000,6
4,Frenkie de Jong,Midfielder,6.361388,5,5,0.753630,0
5,Ferran Torres,Attacker,5.976543,6,15,0.001871,9
6,Inigo Martinez,Defender,4.670659,7,2,0.973882,-5
7,Dani Olmo,Midfielder,3.874352,8,9,0.500214,1
8,Eric Garcia,Defender,3.654057,9,4,0.842138,-5
9,Alejandro Balde,Defender,2.498473,10,7,0.559887,-3
